# Proliferation example

This notebook introduces `proliferation_model`, the callback that lets cells divide during a simulation. The example uses a healthy density-limited population first, then changes one founder lineage so it ignores the density feedback.

This is a toy example to demonstrate the simulation logic: divisions are sampled once per existing cell per time step, the existing row becomes one daughter, a second daughter row is appended, both daughters copy the parent's post-update U/S counts, and new appended rows become eligible for updates in the next step.

## Preparations

In [ ]:
from matplotlib import colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

from markovmodus import SimulationParameters, SimulationState, simulate_dataset

In [ ]:
# Helper functions.
def num_states_from_adata(adata) -> int:
    return int(np.asarray(adata.uns["state_expression"]).shape[0])


def state_palette(num_states: int) -> list:
    cmap = plt.get_cmap("tab10")
    return [cmap(state % cmap.N) for state in range(num_states)]


def plot_transition_graph(
    matrix: np.ndarray,
    title: str,
    *,
    ax: plt.Axes,
    positions: np.ndarray | None = None,
    palette: list | None = None,
    state_names: list[str] | None = None,
) -> None:
    matrix = np.asarray(matrix, dtype=float)
    num_states = matrix.shape[0]
    if positions is None:
        angles = np.linspace(0, 2 * np.pi, num_states, endpoint=False)
        positions = np.stack((np.cos(angles), np.sin(angles)), axis=1)
    else:
        positions = np.asarray(positions, dtype=float)

    max_rate = float(matrix.max())
    ax.set_title(title, fontsize=13, pad=10)
    ax.set_aspect("equal")
    margin = 0.45
    ax.set_xlim(float(positions[:, 0].min() - margin), float(positions[:, 0].max() + margin))
    ax.set_ylim(float(positions[:, 1].min() - margin), float(positions[:, 1].max() + margin))
    ax.axis("off")

    node_radius = 0.12
    if palette is None:
        palette = state_palette(num_states)
    if state_names is None:
        state_names = [str(state) for state in range(num_states)]
    edge_color = "0.58"

    if max_rate > 0:
        for source in range(num_states):
            for target in range(num_states):
                rate = matrix[source, target]
                if source == target or rate <= 0:
                    continue

                source_pos = positions[source]
                target_pos = positions[target]
                bidirectional = matrix[target, source] > 0
                rad = 0.20 if bidirectional else 0.03
                linewidth = 1.4 + 1.8 * (rate / max_rate)

                ax.annotate(
                    "",
                    xy=target_pos,
                    xytext=source_pos,
                    arrowprops=dict(
                        arrowstyle="-|>",
                        color=edge_color,
                        alpha=0.85,
                        lw=linewidth,
                        mutation_scale=14,
                        shrinkA=18,
                        shrinkB=18,
                        connectionstyle=f"arc3,rad={rad}",
                    ),
                    zorder=1,
                )

                direction = target_pos - source_pos
                normal = np.array([-direction[1], direction[0]])
                norm = np.linalg.norm(normal)
                if norm > 0:
                    normal = normal / norm
                label_position = 0.55 * source_pos + 0.45 * target_pos + normal * rad * 0.55
                ax.text(
                    label_position[0],
                    label_position[1],
                    f"{rate:.2f}",
                    fontsize=8,
                    color="0.35",
                    ha="center",
                    va="center",
                    bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=0.8),
                    zorder=2,
                )

    for state, (x, y) in enumerate(positions):
        node = plt.Circle(
            (x, y),
            node_radius,
            facecolor=palette[state % len(palette)],
            edgecolor="black",
            linewidth=1.2,
            alpha=0.85,
            zorder=3,
        )
        ax.add_patch(node)
        ax.text(
            x,
            y,
            state_names[state],
            ha="center",
            va="center",
            color="white",
            fontsize=11,
            weight="bold",
            zorder=4,
        )


def _normalize_rows_for_plotting(matrix: np.ndarray) -> np.ndarray:
    normalized = np.asarray(matrix, dtype=float).copy()
    row_sums = normalized.sum(axis=1)
    positive_rows = row_sums > 0.0
    normalized[positive_rows] = normalized[positive_rows] / row_sums[positive_rows, None]
    return normalized


def plot_transition_and_division_graph(
    transition_matrix: np.ndarray,
    division_rates: np.ndarray,
    daughter_state_probs: np.ndarray,
    title: str,
    *,
    ax: plt.Axes,
    positions: np.ndarray | None = None,
    palette: list | None = None,
    state_names: list[str] | None = None,
    division_multiplier: float = 1.0,
) -> None:
    transition_matrix = np.asarray(transition_matrix, dtype=float)
    division_rates = np.asarray(division_rates, dtype=float)
    daughter_state_probs = _normalize_rows_for_plotting(daughter_state_probs)
    num_states = transition_matrix.shape[0]

    if positions is None:
        angles = np.linspace(0, 2 * np.pi, num_states, endpoint=False)
        positions = np.stack((np.cos(angles), np.sin(angles)), axis=1)
    else:
        positions = np.asarray(positions, dtype=float)
    if palette is None:
        palette = state_palette(num_states)
    if state_names is None:
        state_names = [str(state) for state in range(num_states)]

    copy_offset = np.array([0.0, -1.35])
    parent_positions = positions
    daughter_positions = positions + copy_offset
    daughter_names = list(state_names)

    ax.set_title(title, fontsize=13, pad=10)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_xlim(float(parent_positions[:, 0].min() - 0.58), float(parent_positions[:, 0].max() + 0.58))
    ax.set_ylim(float(daughter_positions[:, 1].min() - 0.56), float(parent_positions[:, 1].max() + 0.72))

    node_radius = 0.12
    edge_color = "0.58"
    max_transition_rate = float(transition_matrix.max())

    copy_min = daughter_positions.min(axis=0)
    copy_max = daughter_positions.max(axis=0)
    box = Rectangle(
        (copy_min[0] - 0.42, copy_min[1] - 0.40),
        copy_max[0] - copy_min[0] + 0.84,
        copy_max[1] - copy_min[1] + 0.92,
        facecolor="0.98",
        edgecolor="0.75",
        linewidth=1.2,
        zorder=0,
    )
    ax.add_patch(box)
    ax.text(parent_positions[:, 0].mean(), float(parent_positions[:, 1].max() + 0.42), "existing cells", ha="center", fontsize=9, color="0.35")
    ax.text(daughter_positions[:, 0].mean(), float(daughter_positions[:, 1].max() + 0.32), "daughter copy", ha="center", fontsize=9, color="0.35")

    def draw_transition_edges(graph_positions: np.ndarray, *, show_labels: bool) -> None:
        if max_transition_rate <= 0:
            return
        for source in range(num_states):
            for target in range(num_states):
                rate = transition_matrix[source, target]
                if source == target or rate <= 0:
                    continue
                source_pos = graph_positions[source]
                target_pos = graph_positions[target]
                linewidth = 1.2 + 1.7 * (rate / max_transition_rate)
                ax.annotate(
                    "",
                    xy=target_pos,
                    xytext=source_pos,
                    arrowprops=dict(
                        arrowstyle="-|>",
                        color=edge_color,
                        alpha=0.80,
                        lw=linewidth,
                        mutation_scale=13,
                        shrinkA=18,
                        shrinkB=18,
                        connectionstyle="arc3,rad=0.03",
                    ),
                    zorder=1,
                )
                if show_labels:
                    direction = target_pos - source_pos
                    normal = np.array([-direction[1], direction[0]])
                    norm = np.linalg.norm(normal)
                    if norm > 0:
                        normal = normal / norm
                    label_position = 0.55 * source_pos + 0.45 * target_pos + normal * 0.12
                    ax.text(
                        label_position[0],
                        label_position[1],
                        f"{rate:.2f}",
                        fontsize=8,
                        color="0.35",
                        ha="center",
                        va="center",
                        bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=0.8),
                        zorder=2,
                    )

    def draw_nodes(graph_positions: np.ndarray, labels: list[str], *, alpha: float) -> None:
        for state, (x, y) in enumerate(graph_positions):
            node = plt.Circle(
                (x, y),
                node_radius,
                facecolor=palette[state % len(palette)],
                edgecolor="black",
                linewidth=1.2,
                alpha=alpha,
                zorder=4,
            )
            ax.add_patch(node)
            ax.text(
                x,
                y,
                labels[state],
                ha="center",
                va="center",
                color="white",
                fontsize=10,
                weight="bold",
                zorder=5,
            )

    draw_transition_edges(parent_positions, show_labels=True)
    draw_transition_edges(daughter_positions, show_labels=False)

    division_edges = division_multiplier * division_rates[:, None] * daughter_state_probs
    max_division_edge = float(division_edges.max()) if division_edges.size else 0.0
    if max_division_edge > 0.0:
        for source_state in range(num_states):
            for daughter_state in range(num_states):
                edge_rate = float(division_edges[source_state, daughter_state])
                if edge_rate <= 0.0:
                    continue
                source_pos = parent_positions[source_state]
                target_pos = daughter_positions[daughter_state]
                color = palette[source_state % len(palette)]
                linewidth = 1.1 + 3.1 * edge_rate / max_division_edge
                if daughter_state == source_state:
                    curve = 0.0
                else:
                    curve = 0.18 if daughter_state > source_state else -0.18
                ax.annotate(
                    "",
                    xy=target_pos,
                    xytext=source_pos,
                    arrowprops=dict(
                        arrowstyle="-|>",
                        color=color,
                        linestyle="--",
                        alpha=0.82,
                        lw=linewidth,
                        mutation_scale=13,
                        shrinkA=21,
                        shrinkB=21,
                        connectionstyle=f"arc3,rad={curve}",
                    ),
                    zorder=3,
                )
                direction = target_pos - source_pos
                normal = np.array([-direction[1], direction[0]])
                norm = np.linalg.norm(normal)
                if norm > 0:
                    normal = normal / norm
                if normal[0] < 0:
                    normal = -normal
                label_position = 0.55 * source_pos + 0.45 * target_pos + normal * 0.14
                ax.text(
                    label_position[0],
                    label_position[1],
                    f"{edge_rate:.3f}",
                    fontsize=8,
                    color=color,
                    ha="center",
                    va="center",
                    bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=0.8),
                    zorder=4,
                )

    draw_nodes(parent_positions, state_names, alpha=0.9)
    draw_nodes(daughter_positions, daughter_names, alpha=0.65)

    legend_handles = [
        Line2D([0], [0], color=edge_color, lw=2.1, label="latent transition rate"),
        Line2D(
            [0],
            [0],
            color="black",
            lw=2.1,
            linestyle="--",
            label="division into daughter copy",
        ),
    ]
    ax.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.12),
        frameon=False,
        fontsize=8,
        ncol=2,
    )


def preprocess_for_scanpy(adata):
    adata.obs["state"] = adata.obs["state"].astype("category")
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    adata.layers["log_normalized"] = adata.X.copy()

    analysis = adata.copy()
    sc.pp.scale(analysis, max_value=10)
    sc.pp.neighbors(analysis, n_neighbors=min(80, analysis.n_obs - 1), use_rep="X")
    sc.tl.umap(analysis, min_dist=0.95)
    adata.obsm["X_umap"] = analysis.obsm["X_umap"].copy()
    return adata


def state_counts(adata) -> pd.Series:
    counts = adata.obs["state"].astype(int).value_counts().reindex(
        range(num_states_from_adata(adata)), fill_value=0
    )
    counts.index.name = "state"
    return counts


def plot_state_counts(
    adata,
    title: str,
    *,
    ax: plt.Axes,
    palette: list | None = None,
    state_names: list[str] | None = None,
) -> None:
    counts = state_counts(adata)
    if palette is None:
        palette = state_palette(num_states_from_adata(adata))
    if state_names is None:
        state_names = [str(state) for state in counts.index]
    bar_colors = [palette[int(state) % len(palette)] for state in counts.index]
    counts.plot(kind="bar", ax=ax, color=bar_colors)
    ax.set_xlabel("State")
    ax.set_ylabel("Cells")
    ax.set_xticklabels(state_names, rotation=0)
    ax.set_title(title)


def plot_embedding_with_state_labels(
    adata,
    coordinates: np.ndarray,
    title: str,
    *,
    ax: plt.Axes,
    palette: list,
    xlabel: str,
    ylabel: str,
    state_names: list[str] | None = None,
) -> None:
    states = adata.obs["state"].astype(int).to_numpy()
    if state_names is None:
        state_names = [str(state) for state in range(num_states_from_adata(adata))]
    for state in range(num_states_from_adata(adata)):
        mask = states == state
        if not mask.any():
            continue
        color = palette[state % len(palette)]
        ax.scatter(
            coordinates[mask, 0],
            coordinates[mask, 1],
            s=18,
            alpha=1.0,
            color=color,
            linewidths=0,
            zorder=1,
        )
        center = coordinates[mask].mean(axis=0)
        ax.scatter(
            center[0],
            center[1],
            s=520,
            color=color,
            edgecolor="black",
            linewidth=1.2,
            alpha=0.7,
            zorder=3,
        )
        ax.text(
            center[0],
            center[1],
            state_names[state],
            ha="center",
            va="center",
            color="white",
            fontsize=10,
            weight="bold",
            zorder=4,
        )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)


def plot_umap_view(
    adata,
    title: str,
    *,
    ax: plt.Axes,
    palette: list,
    state_names: list[str] | None = None,
) -> None:
    plot_embedding_with_state_labels(
        adata,
        adata.obsm["X_umap"],
        title,
        ax=ax,
        palette=palette,
        xlabel="UMAP1",
        ylabel="UMAP2",
        state_names=state_names,
    )


def _state_last_division_time_colors(
    adata,
    palette: list,
    *,
    min_saturation: float = 0.18,
) -> np.ndarray:
    states = adata.obs["state"].astype(int).to_numpy()
    division_time = adata.obs["last_division_time"].to_numpy(dtype=float)
    max_division_time = float(division_time.max()) if division_time.size else 0.0
    if max_division_time > 0.0:
        normalized_division_time = division_time / max_division_time
    else:
        normalized_division_time = np.zeros_like(division_time)

    colors = np.zeros((states.size, 3), dtype=float)
    for state in range(num_states_from_adata(adata)):
        mask = states == state
        if not mask.any():
            continue
        base_rgb = np.array(mcolors.to_rgb(palette[state % len(palette)]))
        base_hsv = mcolors.rgb_to_hsv(base_rgb)
        hsv = np.tile(base_hsv, (int(mask.sum()), 1))
        hsv[:, 1] = min_saturation + (1.0 - min_saturation) * normalized_division_time[mask]
        hsv[:, 2] = np.maximum(hsv[:, 2], 0.82)
        colors[mask] = mcolors.hsv_to_rgb(hsv)
    return colors


def plot_umap_with_division_time_saturation(
    adata,
    title: str,
    *,
    ax: plt.Axes,
    palette: list,
    state_names: list[str] | None = None,
) -> None:
    coordinates = adata.obsm["X_umap"]
    states = adata.obs["state"].astype(int).to_numpy()
    division_time = adata.obs["last_division_time"].to_numpy(dtype=float)
    colors = _state_last_division_time_colors(adata, palette)
    if state_names is None:
        state_names = [str(state) for state in range(num_states_from_adata(adata))]

    ax.scatter(
        coordinates[:, 0],
        coordinates[:, 1],
        s=18,
        alpha=0.95,
        color=colors,
        linewidths=0,
        zorder=1,
    )
    for state in range(num_states_from_adata(adata)):
        mask = states == state
        if not mask.any():
            continue
        center = coordinates[mask].mean(axis=0)
        color = palette[state % len(palette)]
        ax.scatter(
            center[0],
            center[1],
            s=520,
            color=color,
            edgecolor="black",
            linewidth=1.2,
            alpha=0.7,
            zorder=3,
        )
        ax.text(
            center[0],
            center[1],
            state_names[state],
            ha="center",
            va="center",
            color="white",
            fontsize=10,
            weight="bold",
            zorder=4,
        )

    norm = mcolors.Normalize(vmin=float(division_time.min()), vmax=float(division_time.max()))
    mappable = plt.cm.ScalarMappable(norm=norm, cmap="Greys")
    cbar = ax.figure.colorbar(mappable, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("last division time (pale -> saturated)")
    ax.set_title(title)
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")


def _gene_indices(var_names, genes: list[str]) -> list[int]:
    return [int(np.where(var_names == gene)[0][0]) for gene in genes]


def observed_expression_by_state(adata, genes: list[str]) -> np.ndarray:
    gene_indices = _gene_indices(adata.var_names, genes)
    expression = np.asarray(adata.layers["spliced"][:, gene_indices])
    states = adata.obs["state"].astype(int).to_numpy()
    rows = []
    for state_index in range(num_states_from_adata(adata)):
        mask = states == state_index
        if mask.any():
            rows.append(expression[mask].mean(axis=0))
        else:
            rows.append(np.full(len(genes), np.nan))
    return np.vstack(rows)


def plot_configured_and_observed_expression(
    adata,
    genes: list[str],
    *,
    axes: tuple[plt.Axes, plt.Axes],
    state_names: list[str] | None = None,
) -> None:
    configured_indices = _gene_indices(adata.var_names, genes)
    configured = np.asarray(adata.uns["state_expression"], dtype=float)[:, configured_indices]
    observed = observed_expression_by_state(adata, genes)
    state_indices = np.arange(num_states_from_adata(adata))
    if state_names is None:
        state_names = [str(state) for state in state_indices]
    label_step = max(1, len(genes) // 16)
    tick_positions = np.arange(0, len(genes), label_step)

    panels = [
        (axes[0], configured, "Configured expression", "steady-state target"),
        (axes[1], observed, "Observed expression", "mean raw spliced count"),
    ]
    for ax, matrix, title, colorbar_label in panels:
        im = ax.imshow(matrix, aspect="auto", interpolation="nearest", cmap="viridis")
        ax.set_title(title)
        ax.set_yticks(state_indices)
        ax.set_yticklabels(state_names)
        ax.set_ylabel("State")
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([genes[index] for index in tick_positions], rotation=90, fontsize=7)
        ax.set_xlabel("genes")
        ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=colorbar_label)


def population_over_time(adata) -> pd.DataFrame:
    birth_time = adata.obs["birth_time"].to_numpy(dtype=float)
    times = np.sort(np.unique(birth_time))
    return pd.DataFrame(
        {
            "time": times,
            "cells": [np.count_nonzero(birth_time <= time) for time in times],
        }
    )


def descendant_mask(birth_parent: np.ndarray, *, founder: int = 0) -> np.ndarray:
    mask = np.zeros(len(birth_parent), dtype=bool)
    if len(mask) == 0:
        return mask
    mask[founder] = True
    for cell_index, parent in enumerate(birth_parent):
        if parent >= 0 and mask[parent]:
            mask[cell_index] = True
    return mask


def plot_population_growth(adata, label: str, *, ax: plt.Axes) -> None:
    curve = population_over_time(adata)
    ax.step(curve["time"], curve["cells"], where="post", label=label)
    ax.set_xlabel("time")
    ax.set_ylabel("cells")
    ax.set_title("Population size over time")


def plot_clone_embedding(adata, *, ax: plt.Axes) -> None:
    colors = {
        "healthy background": "0.70",
        "density-blind founder clone": "tab:red",
    }
    for label, color in colors.items():
        mask = adata.obs["clone"].to_numpy() == label
        ax.scatter(
            adata.obsm["X_umap"][mask, 0],
            adata.obsm["X_umap"][mask, 1],
            s=18,
            alpha=0.85,
            color=color,
            label=label,
            linewidths=0,
        )
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")
    ax.set_title("Density-blind lineage on UMAP")
    ax.legend(frameon=False)

## Define a density-limited division model

The latent graph has three states: progenitor (`P`), committed (`C`), and target (`T`). Transitions move cells forward. Healthy division is strongest in `P`, weaker in `C`, and absent in `T`.

For division, `proliferation_model` is a function with the signature `Callable[[SimulationState], tuple[np.ndarray, np.ndarray]]`. The part in square brackets, `[SimulationState]`, is the input: the callback receives one `SimulationState` instance at the start of each simulation step. The return annotation after the comma, `tuple[np.ndarray, np.ndarray]`, is the output: the callback must return `(division_rates, daughter_state_probs)` for that step.

`SimulationState` exposes:

- `time`: the current simulation time.
- `cell_states`: the current latent state of every cell.
- `unspliced` and `spliced`: the current U/S count matrices.
- `birth_parent` and `birth_time`: row-creation metadata for appended daughter rows.
- `generation` and `last_division_time`: division-history metadata, which is useful for making division depend on clone history.

The returned arrays in `tuple[np.ndarray, np.ndarray]` can be state-level or per-cell:

- State-level mode: `division_rates.shape == (num_states,)` and `daughter_state_probs.shape == (num_states, num_states)`.
- Per-cell mode: `division_rates.shape == (current_num_cells,)` and `daughter_state_probs.shape == (current_num_cells, num_states)`.

`division_rates` are continuous-time rates. During each fixed-`dt` step, each existing cell can divide with probability `1 - exp(-division_rate * dt)`. Daughter-state probability rows are normalized internally, and rows with a positive division rate must have positive probability mass.

At the start of a step, `transition_rates` and `proliferation_model` are both evaluated from the same `SimulationState`. Existing cells update their U/S counts, then division events are sampled. Dividing cells skip ordinary latent transitions in that step: the existing row becomes daughter 1, one daughter 2 row is appended, and both daughter states are sampled independently from `daughter_state_probs`.

The graph below separates these two processes visually. The upper graph is the current population; the boxed graph below it is the latent graph followed by newly appended daughters. Solid grey arrows are latent transition rates in either copy. Dashed colored arrows show division from a parent state into a daughter state, with labels equal to `division_rate * daughter_state_probability` before density feedback. The feedback curve on the right multiplies those dashed arrows as the target compartment fills.

The healthy model below uses state-level output. Here division rates for each cell are multiplied by a Hill feedback factor that decreases as the target compartment fills.

In [ ]:
P, C, T = 0, 1, 2
NUM_STATES = 3
NUM_GENES = 180
STATE_NAMES = ["P", "C", "T"]
STATE_POSITIONS = np.array([[-1.0, 0.0], [0.0, 0.0], [1.0, 0.0]])
STATE_COLORS = state_palette(NUM_STATES)

transition_rates = np.array(
    [
        [0.0, 0.022, 0.0],
        [0.0, 0.0, 0.030],
        [0.0, 0.0, 0.0],
    ]
)

TARGET_CAPACITY = 100.0
HILL_COEFFICIENT = 3.0
HEALTHY_DIVISION_RATES = np.array([0.05, 0.025, 0.0])
DAUGHTER_STATE_PROBS = np.array(
    [
        [0.75, 0.25, 0.0],
        [0.0, 0.75, 0.25],
        [0.0, 0.0, 1.0],
    ]
)


def target_feedback_from_count(target_count: int) -> float:
    return 1.0 / (1.0 + (target_count / TARGET_CAPACITY) ** HILL_COEFFICIENT)


def target_feedback(state: SimulationState) -> float:
    target_count = int(np.count_nonzero(state.cell_states == T))
    return target_feedback_from_count(target_count)


def healthy_proliferation(state: SimulationState) -> tuple[np.ndarray, np.ndarray]:
    division_rates = HEALTHY_DIVISION_RATES * target_feedback(state)
    return division_rates, DAUGHTER_STATE_PROBS


healthy_params = SimulationParameters(
    # Latent and population dynamics.
    num_states=NUM_STATES,
    transition_rates=transition_rates,
    proliferation_model=healthy_proliferation,

    # State-specific spliced-count targets and U/S dynamics.
    num_genes=NUM_GENES,
    markers_per_state=30,
    marker_overlap={(P, C): 14, (C, T): 14},
    baseline_expression=1.5,
    marker_expression=3.2,
    splicing_rate=0.25,
    decay_rate=0.25,

    # Initial condition.
    initial_state_probs=[1.0, 0.0, 0.0],

    # Simulation size, time, and reproducibility.
    num_cells=60,
    t_final=100.0,
    dt=0.5,
    rng_seed=31,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), gridspec_kw={"width_ratios": [1.0, 1.25]})
plot_transition_and_division_graph(
    transition_rates,
    HEALTHY_DIVISION_RATES,
    DAUGHTER_STATE_PROBS,
    "Transitions plus daughter copy",
    ax=axes[0],
    positions=STATE_POSITIONS,
    palette=STATE_COLORS,
    state_names=STATE_NAMES,
)

target_counts = np.arange(0, 300)
axes[1].plot(target_counts, [target_feedback_from_count(count) for count in target_counts])
axes[1].set_xlabel("target cells")
axes[1].set_ylabel("division-rate multiplier")
axes[1].set_title("Density feedback")
plt.tight_layout()

## Healthy density-limited division

With `proliferation_model` enabled, `num_cells` is the initial population size. The final AnnData object can have more rows because each division reuses one existing row and appends one second-daughter row. Row-creation metadata is stored in `obs["birth_parent"]` and `obs["birth_time"]`; division history is stored in `obs["generation"]` and `obs["last_division_time"]`.

In the UMAP below, hue still encodes the latent state, while saturation encodes `last_division_time`: never-divided cells are pale, and recently divided cells are more saturated.

In [ ]:
healthy_adata = simulate_dataset(healthy_params)
healthy_adata

In [ ]:
healthy_adata.obs[["state", "birth_parent", "birth_time", "generation", "last_division_time"]].head()

In [ ]:
healthy_adata = preprocess_for_scanpy(healthy_adata)
healthy_adata

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
plot_population_growth(healthy_adata, "healthy", ax=axes[0])
plot_state_counts(
    healthy_adata,
    "Final healthy state occupancy",
    ax=axes[1],
    palette=STATE_COLORS,
    state_names=STATE_NAMES,
)
plot_umap_with_division_time_saturation(
    healthy_adata,
    "Healthy UMAP by division time",
    ax=axes[2],
    palette=STATE_COLORS,
    state_names=STATE_NAMES,
)
plt.tight_layout()

## One density-blind founder lineage

The healthy callback returned state-level division rates. To make one individual lineage special, we switch to per-cell output. The callback identifies descendants of cell `0` from `SimulationState.birth_parent`; those cells use a fixed density-blind division rate and a self-renewing daughter-state distribution. All other cells keep the healthy density feedback.

In [ ]:
DENSITY_BLIND_DIVISION_RATE = 0.09
DENSITY_BLIND_DAUGHTER_PROBS = np.array([0.88, 0.12, 0.0])


def density_blind_founder_proliferation(
    state: SimulationState,
) -> tuple[np.ndarray, np.ndarray]:
    feedback = target_feedback(state)
    division_rates = (HEALTHY_DIVISION_RATES * feedback)[state.cell_states]
    daughter_probs = DAUGHTER_STATE_PROBS[state.cell_states].copy()

    founder_clone = descendant_mask(state.birth_parent, founder=0)
    division_rates[founder_clone] = DENSITY_BLIND_DIVISION_RATE
    daughter_probs[founder_clone] = DENSITY_BLIND_DAUGHTER_PROBS
    return division_rates, daughter_probs


cancer_params = SimulationParameters(
    # Latent and population dynamics.
    num_states=NUM_STATES,
    transition_rates=transition_rates,
    proliferation_model=density_blind_founder_proliferation,

    # State-specific spliced-count targets and U/S dynamics.
    num_genes=NUM_GENES,
    markers_per_state=30,
    marker_overlap={(P, C): 14, (C, T): 14},
    baseline_expression=1.5,
    marker_expression=3.2,
    splicing_rate=0.25,
    decay_rate=0.25,

    # Initial condition.
    initial_state_probs=[1.0, 0.0, 0.0],

    # Simulation size, time, and reproducibility.
    num_cells=60,
    t_final=100.0,
    dt=0.5,
    rng_seed=31,
)

In [ ]:
cancer_adata = simulate_dataset(cancer_params)
clone = descendant_mask(cancer_adata.obs["birth_parent"].to_numpy(dtype=int), founder=0)
cancer_adata.obs["clone"] = pd.Categorical(
    np.where(clone, "density-blind founder clone", "healthy background"),
    categories=["healthy background", "density-blind founder clone"],
)
cancer_adata = preprocess_for_scanpy(cancer_adata)
cancer_adata

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
plot_population_growth(healthy_adata, "healthy", ax=axes[0])
plot_population_growth(cancer_adata, "density-blind founder", ax=axes[0])
axes[0].legend(frameon=False)

plot_state_counts(
    cancer_adata,
    "Final state occupancy",
    ax=axes[1],
    palette=STATE_COLORS,
    state_names=STATE_NAMES,
)

clone_counts = pd.crosstab(cancer_adata.obs["state"].astype(int), cancer_adata.obs["clone"])
clone_counts = clone_counts.reindex(range(NUM_STATES), fill_value=0)
clone_counts.index = STATE_NAMES
clone_counts.plot(kind="bar", stacked=True, ax=axes[2], color=["0.70", "tab:red"])
axes[2].set_xlabel("State")
axes[2].set_ylabel("Cells")
axes[2].set_title("Founder lineage by state")
axes[2].legend(frameon=False)

plot_umap_with_division_time_saturation(
    cancer_adata,
    "Founder UMAP by division time",
    ax=axes[3],
    palette=STATE_COLORS,
    state_names=STATE_NAMES,
)

plot_clone_embedding(cancer_adata, ax=axes[4])
plt.tight_layout()